# LeetCode 448 — Find All Numbers Disappeared in an Array bằng Python

**Link đề:** https://leetcode.com/problems/find-all-numbers-disappeared-in-an-array/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, kỹ thuật "đánh dấu tại chỗ" (in-place marking)

```python
class Solution:
    def findDisappearedNumbers(self, nums: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu đề: mảng dài `n`, mọi giá trị nằm trong `[1, n]`, tìm các số **không xuất hiện**.
2. Ôn `in`, `set`, `list` đánh dấu (mảng boolean), `range`, `abs`, list comprehension.
3. Đi từ cách trực giác `O(n²)` → dùng `set` `O(n)` → mảng đánh dấu → **đánh dấu bằng dấu âm ngay trong `nums`** (bộ nhớ phụ `O(1)`, đúng follow-up) → cyclic sort.
4. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, nhận dạng pattern "giá trị chính là chỉ số".
5. Tự làm 5 bài tập từ dễ đến khó, có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài — số nào bị "biến mất"?

Cho mảng `nums` gồm `n` số nguyên, mỗi `nums[i]` nằm trong đoạn `[1, n]`. Trả về **danh sách tất cả các số trong `[1, n]` không xuất hiện** trong `nums`.

| Input `nums` | `n` | Các số phải có | Output | Giải thích |
|---|:---:|---|---|---|
| `[4,3,2,7,8,2,3,1]` | 8 | 1..8 | `[5,6]` | 2 và 3 xuất hiện hai lần, "chiếm chỗ" của 5 và 6 |
| `[1,1]` | 2 | 1..2 | `[2]` | 1 lặp lại, 2 vắng mặt |

**Constraints:** `n == nums.length`, `1 <= n <= 10^5`, `1 <= nums[i] <= n`.

**Follow up:** Làm được trong `O(n)` thời gian và **không dùng bộ nhớ phụ** (list kết quả không tính) không?

Quan sát quan trọng: có `n` ô và `n` giá trị có thể. Nếu có số nào lặp lại thì chắc chắn có số khác vắng mặt — số lần lặp "dư ra" bằng đúng số lượng số vắng mặt.

In [1]:
nums = [4, 3, 2, 7, 8, 2, 3, 1]
n = len(nums)
print("n =", n)
print("Các số cần có:", list(range(1, n + 1)))
print("Mảng đã sắp xếp:", sorted(nums))

n = 8
Các số cần có: [1, 2, 3, 4, 5, 6, 7, 8]
Mảng đã sắp xếp: [1, 2, 2, 3, 3, 4, 7, 8]


Nhìn mảng đã sắp xếp `[1, 2, 2, 3, 3, 4, 7, 8]` ta thấy ngay: thiếu `5` và `6`. Có 2 số lặp (2 và 3) và 2 số vắng — đúng như quan sát trên.

Câu hỏi đặt ra trước khi code:
- Làm sao kiểm tra "số `x` có trong mảng không" bằng Python?
- Kiểm tra như thế tốn bao nhiêu thời gian? Có nhanh hơn được không?
- Có cách nào **ghi nhớ "đã thấy số `x`"** mà không tạo thêm cấu trúc dữ liệu nào không?

## 2. Nền tảng Python cần biết

### 2.1. `range(1, n + 1)` — sinh các số từ 1 đến n

`range(start, stop)` sinh các số `start, start+1, ..., stop-1` (**không gồm** `stop`). Vì vậy muốn lấy `1..n` phải viết `range(1, n + 1)`.

In [2]:
n = 5
print(list(range(n)))          # 0..4  (bắt đầu từ 0)
print(list(range(1, n + 1)))   # 1..5  (đúng thứ ta cần)
print(list(range(1, n)))       # 1..4  -> thiếu n, lỗi kinh điển!

[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[1, 2, 3, 4]


### 2.2. Toán tử `in` với `list` và với `set`

`x in container` trả về `True/False`. Nhưng **chi phí** khác nhau rất lớn:

- Với `list`: Python dò từng phần tử từ đầu đến cuối → `O(n)`.
- Với `set` (tập hợp): dùng bảng băm (hash table) → trung bình `O(1)`.

`set(nums)` tạo một tập hợp từ list: bỏ phần tử trùng, không giữ thứ tự.

In [3]:
nums = [4, 3, 2, 7, 8, 2, 3, 1]
seen = set(nums)
print("set:", seen)               # các phần tử trùng đã bị gộp
print("5 in nums :", 5 in nums)   # dò cả list
print("5 in seen :", 5 in seen)   # tra bảng băm
print("2 in seen :", 2 in seen)
print("Kích thước list =", len(nums), "| kích thước set =", len(seen))

set: {1, 2, 3, 4, 7, 8}
5 in nums : False
5 in seen : False
2 in seen : True
Kích thước list = 8 | kích thước set = 6


### 2.3. List comprehension — tạo list bằng một dòng

```python
[biểu_thức for x in iterable if điều_kiện]
```

đọc là: "với mỗi `x` trong `iterable`, nếu `điều_kiện` đúng thì đưa `biểu_thức` vào list". Nó tương đương vòng `for` + `append`.

In [4]:
# Cách viết dài
evens = []
for x in range(1, 11):
    if x % 2 == 0:
        evens.append(x)
print(evens)

# Cách viết ngắn bằng list comprehension
print([x for x in range(1, 11) if x % 2 == 0])

[2, 4, 6, 8, 10]
[2, 4, 6, 8, 10]


### 2.4. Chỉ số 0-based và phép "giá trị → chỉ số"

List Python đánh chỉ số từ `0`. Giá trị trong bài lại từ `1` đến `n`. Vậy mỗi giá trị `v` có **một ô riêng** là chỉ số `v - 1`:

| giá trị `v` | 1 | 2 | 3 | ... | n |
|---|---|---|---|---|---|
| chỉ số `v - 1` | 0 | 1 | 2 | ... | n-1 |

Đây là chìa khóa của lời giải tối ưu: ta có thể dùng **chính ô `nums[v-1]`** để ghi lại "đã thấy `v`".

In [5]:
nums = [4, 3, 2, 7, 8, 2, 3, 1]
for v in [1, 4, 8]:
    print(f"giá trị {v} -> ô chỉ số {v - 1} (hiện đang chứa {nums[v - 1]})")

giá trị 1 -> ô chỉ số 0 (hiện đang chứa 4)
giá trị 4 -> ô chỉ số 3 (hiện đang chứa 7)
giá trị 8 -> ô chỉ số 7 (hiện đang chứa 1)


### 2.5. `abs()` và dấu âm như một "lá cờ"

`abs(x)` trả giá trị tuyệt đối: `abs(-7) == 7`. Nếu ta đổi một số dương thành âm, ta **không mất thông tin** về độ lớn của nó (vẫn lấy lại bằng `abs`), nhưng có thêm 1 bit thông tin: "đã được đánh dấu". Đây là mẹo dùng trong cách tối ưu.

In [6]:
x = 7
marked = -x
print("ban đầu:", x, "| sau khi đánh dấu:", marked)
print("độ lớn vẫn còn:", abs(marked))
print("đã đánh dấu chưa?", marked < 0)
# Đánh dấu hai lần vẫn phải giữ âm -> dùng -abs(...) thay vì -x
print("đánh dấu lần 2 sai cách:", -marked, "| đúng cách:", -abs(marked))

ban đầu: 7 | sau khi đánh dấu: -7
độ lớn vẫn còn: 7
đã đánh dấu chưa? True
đánh dấu lần 2 sai cách: 7 | đúng cách: -7


## 3. Cách 1 — Trực giác: thử từng số `1..n` xem có trong list không

Người mới thường nghĩ: "Với mỗi số `x` từ 1 đến n, nếu `x not in nums` thì thêm vào kết quả".

Code rất ngắn, đúng, nhưng `x not in nums` trên **list** tốn `O(n)`, lặp `n` lần → **`O(n²)`**. Với `n = 10^5`, đó là khoảng `10^10` phép so sánh trong trường hợp xấu — quá chậm.

In [7]:
def find_disappeared_bruteforce(nums: list[int]) -> list[int]:
    n = len(nums)
    result = []
    for x in range(1, n + 1):
        if x not in nums:          # dò cả list -> O(n)
            result.append(x)
    return result

print(find_disappeared_bruteforce([4, 3, 2, 7, 8, 2, 3, 1]))  # [5, 6]
print(find_disappeared_bruteforce([1, 1]))                    # [2]

[5, 6]
[2]


### 3.1. Đếm số phép so sánh để thấy nút thắt

Ta tự đếm bằng một biến `comparisons` để thấy chi phí tăng nhanh thế nào khi `n` tăng (dùng mảng xấu nhất: toàn số 1, nên hầu hết `x` phải dò hết list).

In [8]:
def count_comparisons_bruteforce(nums):
    n = len(nums)
    comparisons = 0
    for x in range(1, n + 1):
        for v in nums:              # mô phỏng 'x in nums'
            comparisons += 1
            if v == x:
                break
    return comparisons

for n in [10, 100, 1000]:
    print(f"n = {n:>5} -> số phép so sánh ≈ {count_comparisons_bruteforce([1] * n):>8}")

n =    10 -> số phép so sánh ≈       91
n =   100 -> số phép so sánh ≈     9901
n =  1000 -> số phép so sánh ≈   999001


Số phép so sánh tăng xấp xỉ `n²` (n gấp 10 → công việc gấp ~100). Đó là **nút thắt**: mỗi lần hỏi "x có trong mảng không?" ta lại dò lại từ đầu.

## 4. Cách 2 — Dùng `set` để tra cứu nhanh

Sửa nút thắt: đổ `nums` vào một `set` **một lần** (`O(n)`), sau đó mỗi câu hỏi `x in seen` chỉ tốn trung bình `O(1)`.

In [9]:
def find_disappeared_set(nums: list[int]) -> list[int]:
    seen = set(nums)                                   # O(n) thời gian, O(n) bộ nhớ
    return [x for x in range(1, len(nums) + 1) if x not in seen]

print(find_disappeared_set([4, 3, 2, 7, 8, 2, 3, 1]))  # [5, 6]
print(find_disappeared_set([1, 1]))                    # [2]

[5, 6]
[2]


Độ phức tạp: thời gian `O(n)` trung bình, bộ nhớ phụ `O(n)` cho `set`. Đã đủ nhanh để Accepted, nhưng **chưa đạt follow-up** (không dùng bộ nhớ phụ).

### 4.1. Cách 3 — Mảng đánh dấu (boolean array)

Vì giá trị chỉ nằm trong `1..n`, ta không cần bảng băm: một list `present` dài `n + 1` (bỏ ô 0) là đủ. `present[v] = True` nghĩa là "đã thấy `v`". Cách này nhanh hơn `set` một chút về hằng số và **dẫn đường** cho cách tối ưu: nếu ta có thể dùng chính `nums` thay cho `present` thì không cần bộ nhớ phụ nữa.

In [10]:
def find_disappeared_flags(nums: list[int]) -> list[int]:
    n = len(nums)
    present = [False] * (n + 1)      # [False]*k tạo list k phần tử False
    for v in nums:
        present[v] = True
    return [x for x in range(1, n + 1) if not present[x]]

print(find_disappeared_flags([4, 3, 2, 7, 8, 2, 3, 1]))
print(find_disappeared_flags([1, 1]))

[5, 6]
[2]


## 5. Bước đột phá — dùng chính `nums` làm mảng đánh dấu

Cách 3 cần `n` ô để ghi "đã thấy". Nhưng `nums` **đã có sẵn đúng `n` ô**! Vấn đề: các ô đó đang chứa dữ liệu ta còn cần đọc.

**Mẹo:** ghi "đã thấy `v`" bằng cách **đổi dấu** ô `nums[v - 1]` thành âm. Dấu âm là lá cờ, còn độ lớn (`abs`) vẫn giữ nguyên giá trị gốc để ta đọc tiếp.

Quy trình hai lượt:
1. **Lượt đánh dấu:** với mỗi phần tử, lấy `v = abs(nums[i])` (phải `abs` vì ô này có thể đã bị đánh dấu âm trước đó), rồi đặt `nums[v - 1] = -abs(nums[v - 1])`.
2. **Lượt thu hoạch:** ô `i` nào **vẫn dương** nghĩa là không ai đánh dấu nó → số `i + 1` không xuất hiện.

Thử bằng tay một ví dụ nhỏ `nums = [2, 2, 1]` (n = 3):

| Bước | đọc `v` | ô đánh dấu `v-1` | `nums` sau bước |
|---|---|---|---|
| i=0 | 2 | 1 | `[2, -2, 1]` |
| i=1 | abs(-2)=2 | 1 (đã âm, giữ âm) | `[2, -2, 1]` |
| i=2 | 1 | 0 | `[-2, -2, 1]` |

Ô 2 còn dương → số `3` vắng mặt. Kết quả `[3]`.

In [11]:
nums = [2, 2, 1]
i = 1
v = abs(nums[i])
print("đọc v =", v, "-> đánh dấu ô", v - 1)
nums[v - 1] = -abs(nums[v - 1])
print("nums sau khi đánh dấu:", nums)

đọc v = 2 -> đánh dấu ô 1
nums sau khi đánh dấu: [2, -2, 1]


## 6. Lời giải chính — mổ xẻ từng đoạn rồi ghép lại

### 6.1. Đoạn 1: lượt đánh dấu

```python
for i in range(len(nums)):
    v = abs(nums[i])          # giá trị gốc, bỏ qua lá cờ âm
    idx = v - 1               # ô "nhà" của giá trị v
    nums[idx] = -abs(nums[idx])
```

- Tại sao `abs(nums[i])`? Vì ô `i` có thể đã bị một phần tử đứng trước đánh dấu âm. Nếu dùng thẳng `nums[i]` (âm) thì `idx` sẽ âm và trỏ sai chỗ (Python cho phép chỉ số âm nên **không báo lỗi**, rất nguy hiểm).
- Tại sao `-abs(...)` thay vì `*= -1`? Vì cùng một ô có thể bị đánh dấu nhiều lần (giá trị lặp). Đổi dấu hai lần sẽ biến âm thành dương — mất lá cờ.

In [12]:
def mark_pass(nums):
    for i in range(len(nums)):
        v = abs(nums[i])
        idx = v - 1
        nums[idx] = -abs(nums[idx])
    return nums

print(mark_pass([4, 3, 2, 7, 8, 2, 3, 1]))

[-4, -3, -2, -7, 8, 2, -3, -1]


### 6.2. Đoạn 2: lượt thu hoạch

```python
return [i + 1 for i in range(len(nums)) if nums[i] > 0]
```

Ô `i` còn dương ↔ không có giá trị `i + 1` nào trong mảng. Nhìn output trên: ô 4 (giá trị 8) và ô 5 (giá trị 2) còn dương → số `5` và `6` vắng mặt.

In [13]:
marked = mark_pass([4, 3, 2, 7, 8, 2, 3, 1])
print([i + 1 for i in range(len(marked)) if marked[i] > 0])

[5, 6]


### 6.3. Ghép lại thành hàm hoàn chỉnh

Hàm sau **sửa trực tiếp** `nums`. Đề không cấm điều này, nhưng để không làm hỏng dữ liệu của người gọi trong các demo, ta có thể (tùy chọn) khôi phục dấu ở cuối — chi phí thêm `O(n)` thời gian, vẫn `O(1)` bộ nhớ.

In [14]:
def find_disappeared_inplace(nums: list[int]) -> list[int]:
    # Lượt 1: đánh dấu ô "nhà" của mỗi giá trị bằng dấu âm
    for i in range(len(nums)):
        idx = abs(nums[i]) - 1
        nums[idx] = -abs(nums[idx])
    # Lượt 2: ô còn dương -> số i+1 vắng mặt
    result = [i + 1 for i in range(len(nums)) if nums[i] > 0]
    # (Tùy chọn) khôi phục mảng về như cũ
    for i in range(len(nums)):
        nums[i] = abs(nums[i])
    return result

data = [4, 3, 2, 7, 8, 2, 3, 1]
print(find_disappeared_inplace(data))
print("mảng đã được khôi phục:", data)
print(find_disappeared_inplace([1, 1]))

[5, 6]
mảng đã được khôi phục: [4, 3, 2, 7, 8, 2, 3, 1]
[2]


## 7. TRACE từng bước — nhìn lá cờ âm lan ra

Bảng trace với `nums = [4,3,2,7,8,2,3,1]` (chỉ lượt đánh dấu):

| i | `nums[i]` | `v = abs` | ô đánh dấu | ô đó trước → sau | `nums` sau bước |
|---|---|---|---|---|---|
| 0 | 4 | 4 | 3 | 7 → -7 | `[4,3,2,-7,8,2,3,1]` |
| 1 | 3 | 3 | 2 | 2 → -2 | `[4,3,-2,-7,8,2,3,1]` |
| 2 | -2 | 2 | 1 | 3 → -3 | `[4,-3,-2,-7,8,2,3,1]` |
| 3 | -7 | 7 | 6 | 3 → -3 | `[4,-3,-2,-7,8,2,-3,1]` |
| 4 | 8 | 8 | 7 | 1 → -1 | `[4,-3,-2,-7,8,2,-3,-1]` |
| 5 | 2 | 2 | 1 | -3 → -3 (giữ) | không đổi |
| 6 | -3 | 3 | 2 | -2 → -2 (giữ) | không đổi |
| 7 | -1 | 1 | 0 | 4 → -4 | `[-4,-3,-2,-7,8,2,-3,-1]` |

Ý nghĩa các cột: `nums[i]` có thể âm (đã bị đánh dấu trước), nên luôn đọc qua `abs`. Hàng 5 và 6 là **nhánh đặc biệt**: giá trị lặp → ô đã âm, `-abs` giữ nguyên âm. Cuối cùng ô 4 và 5 dương → `[5, 6]`.

In [15]:
def trace_find_disappeared(nums):
    nums = list(nums)   # sao chép để không phá input gốc
    print("Bắt đầu:", nums)
    for i in range(len(nums)):
        cur = nums[i]
        v = abs(cur)
        idx = v - 1
        before = nums[idx]
        nums[idx] = -abs(nums[idx])
        note = "đã âm, giữ nguyên" if before < 0 else "đổi sang âm"
        print(f"i={i}: nums[i]={cur:>3}, v={v}, ô {idx}: {before:>3} -> {nums[idx]:>3} ({note}) | {nums}")
    result = [i + 1 for i in range(len(nums)) if nums[i] > 0]
    print("Ô còn dương:", [i for i in range(len(nums)) if nums[i] > 0], "-> kết quả", result)
    return result

trace_find_disappeared([4, 3, 2, 7, 8, 2, 3, 1])

Bắt đầu: [4, 3, 2, 7, 8, 2, 3, 1]
i=0: nums[i]=  4, v=4, ô 3:   7 ->  -7 (đổi sang âm) | [4, 3, 2, -7, 8, 2, 3, 1]
i=1: nums[i]=  3, v=3, ô 2:   2 ->  -2 (đổi sang âm) | [4, 3, -2, -7, 8, 2, 3, 1]
i=2: nums[i]= -2, v=2, ô 1:   3 ->  -3 (đổi sang âm) | [4, -3, -2, -7, 8, 2, 3, 1]
i=3: nums[i]= -7, v=7, ô 6:   3 ->  -3 (đổi sang âm) | [4, -3, -2, -7, 8, 2, -3, 1]
i=4: nums[i]=  8, v=8, ô 7:   1 ->  -1 (đổi sang âm) | [4, -3, -2, -7, 8, 2, -3, -1]
i=5: nums[i]=  2, v=2, ô 1:  -3 ->  -3 (đã âm, giữ nguyên) | [4, -3, -2, -7, 8, 2, -3, -1]
i=6: nums[i]= -3, v=3, ô 2:  -2 ->  -2 (đã âm, giữ nguyên) | [4, -3, -2, -7, 8, 2, -3, -1]
i=7: nums[i]= -1, v=1, ô 0:   4 ->  -4 (đổi sang âm) | [-4, -3, -2, -7, 8, 2, -3, -1]
Ô còn dương: [4, 5] -> kết quả [5, 6]


[5, 6]

### 7.1. Trace thêm ca biên: mảng toàn số giống nhau

`[1, 1, 1, 1]`: chỉ ô 0 bị đánh dấu (nhiều lần), các ô 1, 2, 3 còn dương → `[2, 3, 4]`.

In [16]:
trace_find_disappeared([1, 1, 1, 1])
print()
trace_find_disappeared([1])        # n = 1, không thiếu gì

Bắt đầu: [1, 1, 1, 1]
i=0: nums[i]=  1, v=1, ô 0:   1 ->  -1 (đổi sang âm) | [-1, 1, 1, 1]
i=1: nums[i]=  1, v=1, ô 0:  -1 ->  -1 (đã âm, giữ nguyên) | [-1, 1, 1, 1]
i=2: nums[i]=  1, v=1, ô 0:  -1 ->  -1 (đã âm, giữ nguyên) | [-1, 1, 1, 1]
i=3: nums[i]=  1, v=1, ô 0:  -1 ->  -1 (đã âm, giữ nguyên) | [-1, 1, 1, 1]
Ô còn dương: [1, 2, 3] -> kết quả [2, 3, 4]

Bắt đầu: [1]
i=0: nums[i]=  1, v=1, ô 0:   1 ->  -1 (đổi sang âm) | [-1]
Ô còn dương: [] -> kết quả []


[]

## 8. Cách 4 (sáng tạo) — Cyclic sort: đặt mỗi số về "nhà" của nó

Một ý tưởng khác cũng dùng `O(1)` bộ nhớ: **hoán đổi** để số `v` về đúng ô `v - 1`. Duyệt `i`; trong khi `nums[i]` chưa ở nhà **và** nhà của nó chưa có đúng số đó, ta swap. Sau khi xong, ô `i` nào chứa giá trị khác `i + 1` thì `i + 1` vắng mặt.

Cú pháp hoán đổi của Python: `a, b = b, a` (vế phải được tính xong trước rồi mới gán). Khi hoán đổi phần tử trong list, nên tính chỉ số ra biến riêng trước để tránh nhầm thứ tự gán.

In [17]:
a, b2 = 1, 2
a, b2 = b2, a
print("sau swap:", a, b2)

def find_disappeared_cyclic(nums: list[int]) -> list[int]:
    nums = list(nums)  # demo: làm trên bản sao
    i = 0
    while i < len(nums):
        home = nums[i] - 1
        if nums[i] != nums[home]:          # nhà chưa có đúng số này -> đổi chỗ
            nums[i], nums[home] = nums[home], nums[i]
        else:                              # đã ở nhà, hoặc nhà đã có bản sao -> sang ô kế
            i += 1
    return [i + 1 for i in range(len(nums)) if nums[i] != i + 1]

print(find_disappeared_cyclic([4, 3, 2, 7, 8, 2, 3, 1]))
print(find_disappeared_cyclic([1, 1]))

sau swap: 2 1
[5, 6]
[2]


Vì sao vòng `while` không chạy mãi? Mỗi lần swap đưa **ít nhất một số về đúng nhà vĩnh viễn**, nên tổng số swap ≤ `n`; cộng thêm `n` lần `i += 1` → tổng `O(n)`.

Trace ngắn cyclic sort trên `[4,3,2,7,8,2,3,1]`:

In [18]:
def trace_cyclic(nums):
    nums = list(nums)
    i = 0
    while i < len(nums):
        home = nums[i] - 1
        if nums[i] != nums[home]:
            print(f"i={i}: {nums[i]} chưa ở nhà (ô {home} đang là {nums[home]}) -> swap")
            nums[i], nums[home] = nums[home], nums[i]
            print("      ", nums)
        else:
            i += 1
    print("Sau khi xếp:", nums)
    return [i + 1 for i in range(len(nums)) if nums[i] != i + 1]

print("Kết quả:", trace_cyclic([4, 3, 2, 7, 8, 2, 3, 1]))

i=0: 4 chưa ở nhà (ô 3 đang là 7) -> swap
       [7, 3, 2, 4, 8, 2, 3, 1]
i=0: 7 chưa ở nhà (ô 6 đang là 3) -> swap
       [3, 3, 2, 4, 8, 2, 7, 1]
i=0: 3 chưa ở nhà (ô 2 đang là 2) -> swap
       [2, 3, 3, 4, 8, 2, 7, 1]
i=0: 2 chưa ở nhà (ô 1 đang là 3) -> swap
       [3, 2, 3, 4, 8, 2, 7, 1]
i=4: 8 chưa ở nhà (ô 7 đang là 1) -> swap
       [3, 2, 3, 4, 1, 2, 7, 8]
i=4: 1 chưa ở nhà (ô 0 đang là 3) -> swap
       [1, 2, 3, 4, 3, 2, 7, 8]
Sau khi xếp: [1, 2, 3, 4, 3, 2, 7, 8]
Kết quả: [5, 6]


## 9. Code nộp LeetCode (Python 3)

Bản chính: đánh dấu bằng dấu âm (đạt follow-up). LeetCode cho phép sửa `nums`, nên không cần khôi phục.

In [19]:
class Solution:
    def findDisappearedNumbers(self, nums: list[int]) -> list[int]:
        for i in range(len(nums)):
            idx = abs(nums[i]) - 1
            nums[idx] = -abs(nums[idx])
        return [i + 1 for i in range(len(nums)) if nums[i] > 0]

sol = Solution()
print(sol.findDisappearedNumbers([4, 3, 2, 7, 8, 2, 3, 1]))  # [5, 6]
print(sol.findDisappearedNumbers([1, 1]))                    # [2]

[5, 6]
[2]


Bản ngắn dùng `set` (dễ nhớ, `O(n)` bộ nhớ) — cũng Accepted:

```python
class Solution:
    def findDisappearedNumbers(self, nums: list[int]) -> list[int]:
        seen = set(nums)
        return [x for x in range(1, len(nums) + 1) if x not in seen]
```

## 10. Vì sao thuật toán đúng? (invariant)

**Invariant sau lượt đánh dấu:** `nums[j] < 0` **khi và chỉ khi** giá trị `j + 1` xuất hiện trong mảng gốc.

- Chiều "⇐": nếu `j + 1` xuất hiện ở vị trí `i`, khi vòng lặp tới `i` ta đọc `abs(nums[i]) = j + 1` (vì `abs` khôi phục giá trị gốc) và đặt `nums[j]` âm. Sau đó không ai đổi nó về dương, vì ta chỉ dùng `-abs(...)`.
- Chiều "⇒": ô `j` chỉ bị gán âm khi đọc được giá trị `j + 1`. Mà các giá trị đọc được (qua `abs`) chính là các giá trị gốc → `j + 1` có trong mảng.
- Thêm nữa: đổi dấu không làm mất độ lớn, nên mọi lần đọc `abs(nums[i])` đều ra đúng giá trị gốc của ô `i`.

Do đó lượt thu hoạch lấy đúng **và đủ** các số vắng mặt, theo thứ tự tăng dần.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force `x not in nums` | `O(n²)` | `O(1)` | TLE khi `n = 10^5` |
| 2. `set` | `O(n)` trung bình | `O(n)` | ngắn, dễ nhớ |
| 3. Mảng boolean | `O(n)` | `O(n)` | không cần băm |
| 4. Đánh dấu âm tại chỗ | `O(n)` | `O(1)` | đạt follow-up, sửa input |
| 5. Cyclic sort | `O(n)` | `O(1)` | đạt follow-up, xáo trộn input |

(List kết quả không tính vào bộ nhớ phụ theo đề.)

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Ta tạo hàm `check` gọn và chạy mọi cách trên cùng bộ input. Lưu ý: các cách sửa input phải được gọi với **bản sao** (`list(nums)`) để không ảnh hưởng nhau.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

cases = [
    ("đề 1", [4, 3, 2, 7, 8, 2, 3, 1], [5, 6]),
    ("đề 2", [1, 1], [2]),
    ("n = 1", [1], []),
    ("đủ cả, xáo trộn", [3, 1, 2], []),
    ("toàn số n", [4, 4, 4, 4], [1, 2, 3]),
    ("toàn số 1", [1, 1, 1], [2, 3]),
    ("lặp ở đầu và cuối", [2, 2, 3, 3], [1, 4]),
]
sol = Solution()
for name, nums, expected in cases:
    check(name, sol.findDisappearedNumbers(list(nums)), expected)

PASS  đề 1: [5, 6]
PASS  đề 2: [2]
PASS  n = 1: []
PASS  đủ cả, xáo trộn: []
PASS  toàn số n: [1, 2, 3]
PASS  toàn số 1: [2, 3]
PASS  lặp ở đầu và cuối: [1, 4]


In [21]:
import random
random.seed(448)
methods = {
    "bruteforce": find_disappeared_bruteforce,
    "set": find_disappeared_set,
    "flags": find_disappeared_flags,
    "inplace": find_disappeared_inplace,
    "cyclic": find_disappeared_cyclic,
}
for trial in range(300):
    n = random.randint(1, 12)
    nums = [random.randint(1, n) for _ in range(n)]
    expected = find_disappeared_bruteforce(list(nums))
    for name, f in methods.items():
        got = f(list(nums))
        assert got == expected, (name, nums, got, expected)
print("300 bộ ngẫu nhiên: 5 cách cho kết quả giống hệt nhau")

300 bộ ngẫu nhiên: 5 cách cho kết quả giống hệt nhau


Tính chất phụ đáng kiểm: **số lượng số vắng mặt = n − số giá trị phân biệt**. Đây là một "oracle" (cách kiểm chứng độc lập) rẻ tiền.

In [22]:
random.seed(1)
for _ in range(200):
    n = random.randint(1, 20)
    nums = [random.randint(1, n) for _ in range(n)]
    assert len(Solution().findDisappearedNumbers(list(nums))) == n - len(set(nums))
print("Tính chất |kết quả| = n - |set(nums)| đúng trên 200 bộ")

Tính chất |kết quả| = n - |set(nums)| đúng trên 200 bộ


## 13. Lỗi người mới hay mắc — code sai, vì sao sai, cách sửa

### Lỗi 1: quên `abs` khi đọc giá trị

```python
for i in range(len(nums)):
    idx = nums[i] - 1          # SAI: nums[i] có thể đã âm
    nums[idx] = -abs(nums[idx])
```

Nếu `nums[i] = -3` thì `idx = -4`, Python hiểu là "ô thứ 4 tính từ cuối" — **không báo lỗi** mà đánh dấu nhầm ô.

In [23]:
def wrong_no_abs(nums):
    nums = list(nums)
    for i in range(len(nums)):
        idx = nums[i] - 1
        nums[idx] = -abs(nums[idx])
    return [i + 1 for i in range(len(nums)) if nums[i] > 0]

print("sai :", wrong_no_abs([4, 3, 2, 7, 8, 2, 3, 1]))
print("đúng:", find_disappeared_inplace([4, 3, 2, 7, 8, 2, 3, 1]))

sai : [2, 5]
đúng: [5, 6]


### Lỗi 2: dùng `nums[idx] *= -1` (lật dấu) thay vì `-abs`

Với giá trị lặp, ô bị lật hai lần → quay về dương → bị báo nhầm là "vắng mặt".

In [24]:
def wrong_flip(nums):
    nums = list(nums)
    for i in range(len(nums)):
        idx = abs(nums[i]) - 1
        nums[idx] *= -1            # SAI: lật dấu
    return [i + 1 for i in range(len(nums)) if nums[i] > 0]

print("sai :", wrong_flip([1, 1]))       # báo 1 vắng mặt là sai
print("đúng:", find_disappeared_inplace([1, 1]))

sai : [1, 2]
đúng: [2]


### Lỗi 3: `range(1, n)` thay vì `range(1, n + 1)`

Bỏ sót số `n`. Ví dụ `[1, 1]` phải trả `[2]`, nhưng `range(1, 2)` chỉ thử số 1 → trả `[]`.

### Lỗi 4: trả về chỉ số thay vì giá trị

Thu hoạch `[i for i ... if nums[i] > 0]` cho ra `[4, 5]` thay vì `[5, 6]`. Nhớ `+ 1` để đổi từ chỉ số 0-based sang giá trị.

### Lỗi 5: tưởng `set` là đạt follow-up

`set(nums)` tốn `O(n)` bộ nhớ phụ. Follow-up yêu cầu `O(1)` phụ → phải dùng đánh dấu tại chỗ hoặc cyclic sort.

In [25]:
def wrong_range(nums):
    seen = set(nums)
    return [x for x in range(1, len(nums)) if x not in seen]   # SAI: thiếu n

print("sai :", wrong_range([1, 1]))
print("đúng:", find_disappeared_set([1, 1]))

sai : []
đúng: [2]


## 14. Mẹo nhận dạng pattern

Dấu hiệu "**giá trị nằm trong `[1, n]` (hoặc `[0, n-1]`) với mảng dài `n`**" gần như luôn gợi ý:

- Dùng **giá trị làm chỉ số** (`v - 1`) → thay hash map bằng chính mảng.
- Đánh dấu bằng **dấu âm**, hoặc cộng thêm `n` (`nums[v-1] += n`), hoặc **cyclic sort**.

Các bài cùng họ:
- 442 Find All Duplicates in an Array (đánh dấu âm, gặp ô đã âm → trùng).
- 268 Missing Number (thiếu đúng 1 số trong `[0, n]` → tổng hoặc XOR).
- 645 Set Mismatch (một số lặp, một số thiếu).
- 41 First Missing Positive (Hard, cyclic sort).
- 287 Find the Duplicate Number.

## 15. Biến thể

### 15.1. Tìm các số xuất hiện **hai lần** (LeetCode 442)

Cùng lượt đánh dấu, nhưng nếu khi tới ô `idx` mà nó **đã âm** thì `idx + 1` đã được thấy trước đó → là số trùng.

In [26]:
def find_duplicates(nums):
    nums = list(nums)
    dups = []
    for i in range(len(nums)):
        idx = abs(nums[i]) - 1
        if nums[idx] < 0:
            dups.append(idx + 1)
        else:
            nums[idx] = -nums[idx]
    return dups

print(find_duplicates([4, 3, 2, 7, 8, 2, 3, 1]))  # [2, 3]

[2, 3]


### 15.2. Đánh dấu bằng cách cộng `n` thay cho dấu âm

Thay vì đổi dấu, cộng `n` vào ô nhà: `nums[(v-1)] += n`, đọc giá trị gốc bằng `(nums[i] - 1) % n + 1`. Ô nào còn `<= n` → số vắng. Mẹo này hữu ích khi mảng có thể chứa số 0 (đổi dấu 0 không tạo khác biệt).

In [27]:
def find_disappeared_add_n(nums):
    nums = list(nums)
    n = len(nums)
    for i in range(n):
        v = (nums[i] - 1) % n + 1      # khôi phục giá trị gốc 1..n
        nums[v - 1] += n
    return [i + 1 for i in range(n) if nums[i] <= n]

print(find_disappeared_add_n([4, 3, 2, 7, 8, 2, 3, 1]))
print(find_disappeared_add_n([1, 1]))

[5, 6]
[2]


## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO. Các test để ở dạng comment; tự bỏ comment khi làm xong. Đừng xem đáp án ở cuối trước khi thử!

### Bài 1 — Tính tay
Với `nums = [3, 3, 1, 1]`, lập bảng trace lượt đánh dấu âm (cột `i`, `v`, ô đánh dấu, `nums` sau bước). Kết quả cuối là gì?

### Bài 2 — Hoàn thiện cách dùng `set`

In [28]:
def ex2_missing_with_set(nums: list[int]) -> list[int]:
    # TODO 1: tạo set từ nums
    # TODO 2: duyệt x từ 1 đến n (nhớ n + 1)
    # TODO 3: giữ x nếu x không có trong set
    pass

# assert ex2_missing_with_set([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
# assert ex2_missing_with_set([1, 1]) == [2]

### Bài 3 — Tự viết lời giải đánh dấu âm (không dùng `set`, không tạo list phụ ngoài kết quả)

In [29]:
def ex3_missing_inplace(nums: list[int]) -> list[int]:
    # TODO: lượt 1 đánh dấu, lượt 2 thu hoạch
    pass

# assert ex3_missing_inplace([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
# assert ex3_missing_inplace([2, 2]) == [1]

### Bài 4 — Biến thể Set Mismatch (LeetCode 645)
Mảng `nums` lẽ ra chứa đúng `1..n`, nhưng một số bị **nhân đôi** thay cho một số khác. Trả về `[số_bị_lặp, số_bị_thiếu]`. Ví dụ `[1, 2, 2, 4]` → `[2, 3]`.

In [30]:
def ex4_set_mismatch(nums: list[int]) -> list[int]:
    # TODO: dùng kỹ thuật đánh dấu âm để tìm cả số lặp và số thiếu
    pass

# assert ex4_set_mismatch([1, 2, 2, 4]) == [2, 3]
# assert ex4_set_mismatch([1, 1]) == [1, 2]

### Bài 5 — Số dương nhỏ nhất bị thiếu (mini challenge, dạng LeetCode 41)
Mảng `nums` bất kỳ (có thể có số âm, số 0, số lớn hơn n). Tìm số nguyên dương nhỏ nhất không xuất hiện, `O(n)` thời gian. Ví dụ `[3, 4, -1, 1]` → `2`, `[1, 2, 0]` → `3`.

In [31]:
def ex5_first_missing_positive(nums: list[int]) -> int:
    # TODO: chỉ quan tâm các giá trị trong [1, n]; dùng cyclic sort hoặc đánh dấu
    pass

# assert ex5_first_missing_positive([3, 4, -1, 1]) == 2
# assert ex5_first_missing_positive([1, 2, 0]) == 3
# assert ex5_first_missing_positive([7, 8, 9]) == 1

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1**
- Mức 1: luôn đọc `abs(nums[i])`.
- Mức 2: ô bị đánh dấu là `v - 1`.
- Mức 3: chỉ ô 0 và ô 2 bị đánh dấu → ô 1 và ô 3 còn dương.

**Bài 2**
- Mức 1: `set` giúp tra cứu nhanh.
- Mức 2: `range(1, len(nums) + 1)`.
- Mức 3: `[x for x in range(1, n + 1) if x not in seen]`.

**Bài 3**
- Mức 1: giá trị `v` có nhà ở `v - 1`.
- Mức 2: đánh dấu bằng `-abs(...)`.
- Mức 3: thu hoạch `i + 1` với ô `nums[i] > 0`.

**Bài 4**
- Mức 1: số lặp là số mà khi đánh dấu thì ô nhà **đã âm**.
- Mức 2: số thiếu là ô còn dương sau lượt đánh dấu.
- Mức 3: một vòng `for` vừa đánh dấu vừa bắt số lặp, rồi một vòng tìm ô dương.

**Bài 5**
- Mức 1: đáp án luôn nằm trong `[1, n + 1]`.
- Mức 2: cyclic sort chỉ đặt `v` về nhà khi `1 <= v <= n`.
- Mức 3: sau khi xếp, ô đầu tiên `i` có `nums[i] != i + 1` → trả `i + 1`; nếu không có → `n + 1`.

## 17. Đáp án tham khảo (đặt cuối để tránh lộ trước)

**Bài 1:** `[3,3,1,1]` → i=0: v=3, ô 2: 1→-1 → `[3,3,-1,1]`; i=1: v=3, ô 2 giữ -1; i=2: v=1, ô 0: 3→-3 → `[-3,3,-1,1]`; i=3: v=1, ô 0 giữ. Ô 1 và 3 dương → `[2, 4]`.

In [32]:
def ans2_missing_with_set(nums):
    seen = set(nums)
    return [x for x in range(1, len(nums) + 1) if x not in seen]

def ans3_missing_inplace(nums):
    for i in range(len(nums)):
        idx = abs(nums[i]) - 1
        nums[idx] = -abs(nums[idx])
    return [i + 1 for i in range(len(nums)) if nums[i] > 0]

def ans4_set_mismatch(nums):
    nums = list(nums)
    dup = -1
    for i in range(len(nums)):
        idx = abs(nums[i]) - 1
        if nums[idx] < 0:
            dup = idx + 1
        else:
            nums[idx] = -nums[idx]
    missing = next(i + 1 for i in range(len(nums)) if nums[i] > 0)
    return [dup, missing]

def ans5_first_missing_positive(nums):
    nums = list(nums)
    n = len(nums)
    i = 0
    while i < n:
        v = nums[i]
        if 1 <= v <= n and nums[v - 1] != v:
            nums[i], nums[v - 1] = nums[v - 1], nums[i]
        else:
            i += 1
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1

assert ans3_missing_inplace([3, 3, 1, 1]) == [2, 4]          # bài 1
assert ans2_missing_with_set([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
assert ans2_missing_with_set([1, 1]) == [2]
assert ans3_missing_inplace([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
assert ans3_missing_inplace([2, 2]) == [1]
assert ans4_set_mismatch([1, 2, 2, 4]) == [2, 3]
assert ans4_set_mismatch([1, 1]) == [1, 2]
assert ans5_first_missing_positive([3, 4, -1, 1]) == 2
assert ans5_first_missing_positive([1, 2, 0]) == 3
assert ans5_first_missing_positive([7, 8, 9]) == 1
print("Đáp án bài 1–5: PASS toàn bộ")

Đáp án bài 1–5: PASS toàn bộ


Giải thích `next(...)` trong đáp án bài 4: `next(generator)` lấy **phần tử đầu tiên** mà generator sinh ra — ở đây là ô dương đầu tiên. Vì đề đảm bảo có đúng một số thiếu nên luôn tìm thấy.

In [33]:
gen = (x for x in [5, 8, 11] if x > 6)
print(next(gen))   # 8: phần tử đầu tiên thỏa điều kiện

8


## 18. Template nhớ nhanh + checklist submit

```python
# Pattern: giá trị trong [1, n], mảng dài n -> dùng giá trị làm chỉ số
for i in range(len(nums)):
    idx = abs(nums[i]) - 1          # luôn abs khi đọc
    nums[idx] = -abs(nums[idx])     # đánh dấu "đã thấy idx+1", không lật dấu
return [i + 1 for i in range(len(nums)) if nums[i] > 0]
```

**Checklist trước khi submit**
- [ ] Đã dùng `abs(nums[i])` khi đọc giá trị chưa?
- [ ] Đánh dấu bằng `-abs(...)` (không phải `*= -1`) chưa?
- [ ] Trả về `i + 1` (giá trị) chứ không phải `i` (chỉ số)?
- [ ] Nếu dùng `range`, có `n + 1` để gồm cả số `n`?
- [ ] Thử ca `[1]`, `[1, 1]`, mảng toàn một giá trị?
- [ ] Nói được Time `O(n)`, Space phụ `O(1)` và vì sao?